# Scale-free laser self-calibration on the P1 corpus

P1's calibration module states the physics in one line: the in-plane laser angle φ
"is invisible to the dots" and "has to come from a known length (an anchor), or from
the same object seen at two well-separated ranges (scale-free)". P1 implements the
anchor route (`fit_phi_joint`) and uses the scale-free statistic only as a *test*
(`range_trend`). This notebook implements the scale-free route as the **estimator**:
solve φ so that every rigid object on the dive reads the same length at every range,
spending no known length. It is the P4 "Tier 3" closure
(`../../wuwnet-fishsense2026/fishsense_wuwnet/laser.py`, `close_with_apparent_size`)
written in P1's parameterisation so the two can be compared dive by dive.

Why P2 cares: if this works, the laser needs **no target at all** — no slate, no
checkerboard, no brick — only the dive's own rigid objects. That removes two rows of
the barrier table in `HANDOFF.md` §1 for the laser half, and it turns §4.1's
checkerboard-vs-slate confound into an experiment: recalibrate every dive
target-free and see whether the gap survives.

Ground rules from P1 §0 that this notebook obeys:
- known lengths are used **only** to validate, after the fit, never inside it;
- targets are aggregated at p90, never a mean — foreshortening is one-sided;
- per-dive error is reported as an angle.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "imwut_2026_fishsense_lite"))
from fishsense_imwut import calibration as cal

CORPUS = Path.cwd().parent.parent / "imwut_2026_fishsense_lite" / "fish_model_analysis" / "data" / "corpus.csv"
rows = cal.load_rows(CORPUS)
by_dive = cal.group_by_dive(rows)
df = cal.to_frame(rows)
len(rows), len(by_dive)


(2927, 32)

## The estimator

For a candidate φ the laser axis is rotated in the camera–laser plane, every frame's
depth is re-triangulated, and every recorded length is rescaled by the depth ratio
(exact: snout and tail back-project at one depth — P1 §3.3). A rigid object then has
to read one length at every range, but only in its best-presented frames: within
range bins the p90 of length is the pose-free reading, and the Theil–Sen slope of
those p90s against range is the residual. φ is the value that drives that slope to
zero for every model on the dive at once. No known length appears anywhere.


In [2]:
def frame_rays(dive_rows, inv_k):
    """Unit camera rays through each frame's laser dot."""
    px = np.array([[float(x) for x in r["dot"].split(";")] for r in dive_rows])
    d = (inv_k @ np.column_stack([px, np.ones(len(px))]).T).T
    return d / np.linalg.norm(d, axis=1, keepdims=True)


def depths_for_phi(rays, o, a, n, phi):
    """Vectorised `cal.triangulate_depth` for every ray under an in-plane rotation phi."""
    a2 = cal.rotate(a, n, phi); a2 = a2 / np.linalg.norm(a2)
    b = rays @ a2
    d1 = -(rays @ o); e1 = -(a2 @ o)
    den = 1.0 - b * b
    s = (b * e1 - d1) / den; t = (e1 - b * d1) / den
    return 0.5 * (s[:, None] * rays + o + t[:, None] * a2)[:, 2]


def binned_p90_slope(z, length, n_bins=4, min_per_bin=3):
    """Relative Theil-Sen slope (fraction per metre) of the per-bin p90 length against range.
    None when the frames cannot form >= 3 populated bins."""
    order = np.argsort(z); z, length = z[order], length[order]
    edges = np.quantile(z, np.linspace(0, 1, n_bins + 1))
    zs, ls = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (z >= lo) & (z <= hi) if hi == edges[-1] else (z >= lo) & (z < hi)
        if m.sum() >= min_per_bin:
            zs.append(np.median(z[m])); ls.append(np.quantile(length[m], 0.9))
    if len(zs) < 3:
        return None
    slope, _, _ = cal.theil_sen(zs, ls)
    return slope / np.median(ls)


class Dive:
    """One dive's geometry and per-model frame sets, ready to evaluate any phi."""

    def __init__(self, dive_id, dive_rows):
        self.dive_id = dive_id
        self.o, self.a, inv_k, self.n = cal.dive_geometry(dive_rows)
        self.rays = frame_rays(dive_rows, inv_k)
        self.z0 = np.array([float(r["depth_m"]) for r in dive_rows])
        self.length0 = np.array([float(r["length_m"]) for r in dive_rows])
        self.model = np.array([r["model"] for r in dive_rows])
        self.known = np.array([float(r["known_length_m"]) for r in dive_rows])
        for m, metres in cal.MEASURED_REFERENCES_M.items():
            self.known[self.model == m] = metres
        self.rows = dive_rows
        self.baseline = float(np.hypot(*self.o[:2]))

    def usable_models(self):
        out = []
        for m in sorted(set(self.model)):
            k = self.model == m
            z = self.z0[k]
            far = z >= cal.RANGE_TREND_MIN_DEPTH_M
            if far.sum() >= cal.RANGE_TREND_MIN_FRAMES and z[far].max() / z[far].min() >= cal.RANGE_TREND_MIN_RATIO:
                out.append(m)
        return out

    def lengths(self, phi):
        z = depths_for_phi(self.rays, self.o, self.a, self.n, phi)
        return z, self.length0 * z / self.z0

    def scalefree_residual(self, phi, models):
        z, length = self.lengths(phi)
        res = []
        for m in models:
            k = (self.model == m) & (self.z0 >= cal.RANGE_TREND_MIN_DEPTH_M)
            s = binned_p90_slope(z[k], length[k])
            if s is not None:
                res.append(s)
        return np.array(res)

    def fit_scalefree(self, models, span_deg=1.5, step_deg=0.005):
        """phi that zeroes every usable model's scale-free slope, by a fine grid then a local refine."""
        grid = np.deg2rad(np.arange(-span_deg, span_deg + 1e-9, step_deg))
        cost = np.array([np.sum(self.scalefree_residual(p, models) ** 2) for p in grid])
        i = int(np.argmin(cost))
        lo, hi = grid[max(i - 3, 0)], grid[min(i + 3, len(grid) - 1)]
        fine = np.linspace(lo, hi, 61)
        cost_f = np.array([np.sum(self.scalefree_residual(p, models) ** 2) for p in fine])
        return float(fine[int(np.argmin(cost_f))])

    def validation_pct(self, phi, models):
        """Known lengths, used only here: per-model p90 length ratio - 1, in %, median over models."""
        _, length = self.lengths(phi)
        r = [100 * (np.quantile(length[self.model == m] / self.known[self.model == m], 0.9) - 1) for m in models]
        return float(np.median(r))


## Every dive with a usable rigid object

Gates are P1's own (`RANGE_TREND_*`): a model counts if it has ≥ 8 frames beyond 0.8 m
over a ≥ 2× spread of range. The designed foreshortening sessions (`ANGLE_TEST_DIVES`)
are excluded as P1 excludes them: one object at deliberately oblique poses is not a
rigid-object range test.

Three φ per dive: **recorded** (the slate or checkerboard calibration as used in prod,
φ = 0 by definition), **scale-free** (this estimator, no known length), and **anchor**
(P1's `fit_phi_joint`, which drives every model's p90 ratio to its known length — the
best available reference for what φ "should" be). The validation column applies the
known lengths *after* each fit.


In [3]:
records = []
for dive_id in sorted(by_dive):
    if dive_id in cal.ANGLE_TEST_DIVES:
        continue
    d = Dive(dive_id, by_dive[dive_id])
    models = d.usable_models()
    if not models:
        continue
    phi_sf = d.fit_scalefree(models)
    o, a, inv_k, n = cal.dive_geometry(d.rows)
    phi_anchor = cal.fit_phi_joint(d.rows, o, a, inv_k, n)
    z_med = float(np.median(d.z0))
    records.append(dict(
        dive=dive_id, calibrated_from=int(d.rows[0]["calibration_dive_id"]), n=len(d.rows),
        models=len(models), z_med=z_med, baseline_cm=100 * d.baseline,
        recorded_pct=d.validation_pct(0.0, models),
        phi_sf_deg=np.degrees(phi_sf), scalefree_pct=d.validation_pct(phi_sf, models),
        phi_anchor_deg=np.degrees(phi_anchor),
        residual_slope_pct_per_m=100 * float(np.sqrt(np.mean(d.scalefree_residual(phi_sf, models) ** 2))),
    ))
res = pd.DataFrame(records).set_index("dive")
res["sf_minus_anchor_deg"] = res.phi_sf_deg - res.phi_anchor_deg
res["borrowed"] = res.calibrated_from != res.index
pd.set_option("display.width", 200)
print(res.round(3).to_string())


      calibrated_from    n  models  z_med  baseline_cm  recorded_pct  phi_sf_deg  scalefree_pct  phi_anchor_deg  residual_slope_pct_per_m  sf_minus_anchor_deg  borrowed
dive                                                                                                                                                                    
59                 77   86       3  2.108       10.367         1.639       0.119         -2.012           0.040                     0.351                0.080      True
60                 65  103       2  1.482       10.448        -2.107      -0.124          0.123          -0.096                     0.537               -0.028      True
66                 83   51       1  1.793       10.368        -3.016       0.185         -6.924          -0.115                     0.005                0.300      True
76                 63   91       2  1.746       10.435        -3.919       0.081         -5.929          -0.149                     3.672                0.

In [4]:
def summary(r):
    return pd.Series({
        "dives": len(r),
        "|phi_sf - phi_anchor| median (deg)": r.sf_minus_anchor_deg.abs().median(),
        "|phi_sf - phi_anchor| p90 (deg)": r.sf_minus_anchor_deg.abs().quantile(0.9),
        "within 0.05 deg": int((r.sf_minus_anchor_deg.abs() < 0.05).sum()),
        "|recorded error| median (%)": r.recorded_pct.abs().median(),
        "|scale-free error| median (%)": r.scalefree_pct.abs().median(),
        "recorded beyond 2.5 %": int((r.recorded_pct.abs() > 2.5).sum()),
        "scale-free beyond 2.5 %": int((r.scalefree_pct.abs() > 2.5).sum()),
    })

print("all usable dives"); print(summary(res).round(3).to_string()); print()
print("borrowed calibrations only"); print(summary(res[res.borrowed]).round(3).to_string()); print()
print("self-calibrated (checkerboard) only"); print(summary(res[~res.borrowed]).round(3).to_string())


all usable dives
dives                                 22.000
|phi_sf - phi_anchor| median (deg)     0.076
|phi_sf - phi_anchor| p90 (deg)        0.224
within 0.05 deg                        8.000
|recorded error| median (%)            2.175
|scale-free error| median (%)          3.215
recorded beyond 2.5 %                 10.000
scale-free beyond 2.5 %               12.000

borrowed calibrations only
dives                                 18.000
|phi_sf - phi_anchor| median (deg)     0.057
|phi_sf - phi_anchor| p90 (deg)        0.176
within 0.05 deg                        8.000
|recorded error| median (%)            1.873
|scale-free error| median (%)          2.017
recorded beyond 2.5 %                  7.000
scale-free beyond 2.5 %                8.000

self-calibrated (checkerboard) only
dives                                 4.000
|phi_sf - phi_anchor| median (deg)    0.151
|phi_sf - phi_anchor| p90 (deg)       0.344
within 0.05 deg                       0.000
|recorded error| media

## Reading the table

`|phi_sf − phi_anchor|` is the number that matters: how far a calibration that never
saw a known length lands from the one P1 built out of known lengths. P1's own
conversion puts 0.05° at roughly 1 % of length at 2 m.

Taken per dive it is not good: median 0.076°, p90 0.22°, and the scale-free
validation is *worse* than the recorded calibration on median. Two patterns in the
table say why. Single-model dives zero their slope exactly (`residual_slope` ≈ 0) and
still validate badly (66, 501, 492) — the estimator has removed a real range trend
that was not calibration. Multi-model dives cannot zero every model at once
(residuals 0.35–6.7 %/m) — the models disagree about φ. The next section names the
cause and tests it.

`residual_slope` is a diagnostic P1 does not have: φ can only rotate the axis, so a
slope that will not go to zero under any φ means either the baseline `|O|` is wrong —
the "short baseline with a compensating angle" failure P1 found on 498/503/504 — or
the objects carry a range trend of their own.


## Why the models disagree: half-thickness parallax

The single-model dives zero their slope exactly and still validate worse than the
recorded calibration; the multi-model dives cannot zero every model at once. P1 §6.4
names the mechanism: the dot lands on a solid model's **flank** while snout and fork
lie in its **midplane**, so each model reads short by (half thickness / range) — a
per-model `b/z` term (trout −1.8 cm, Snook −2.4, Grouper −5.0, Box −0.4 ≈ 0). A
scale-free estimator on *length* cannot tell that trend from φ: it will rotate the
laser to cancel the model's thickness.

If that is what is happening, φ fitted per model should order by `b`, the Box (a slab,
`b ≈ 0`) should agree with the anchor, and applying P1's parallax correction before
the fit should pull every model onto the anchor. The correction was itself fitted with
known lengths, so this is a **mechanism test, not a calibration** — the calibration
conclusion is in the next cell.


In [5]:
HALF_THICKNESS_B_M = {"Weasly Fish": -0.018, "Snook": -0.024, "Grouper": -0.050, "Box": -0.004}  # P1 §6.4, error = a + b/z

per_model = []
for dive_id in sorted(by_dive):
    if dive_id in cal.ANGLE_TEST_DIVES:
        continue
    d = Dive(dive_id, by_dive[dive_id])
    o, a, inv_k, n = cal.dive_geometry(d.rows)
    phi_anchor = cal.fit_phi_joint(d.rows, o, a, inv_k, n)
    for m in d.usable_models():
        phi_m = d.fit_scalefree([m])
        # parallax-corrected refit: lengthen every frame by b/z before fitting
        if m in HALF_THICKNESS_B_M:
            saved = d.length0.copy()
            k = d.model == m
            d.length0 = saved - np.where(k, HALF_THICKNESS_B_M[m] / d.z0, 0.0)
            phi_m_corr = d.fit_scalefree([m])
            d.length0 = saved
        else:
            phi_m_corr = np.nan
        per_model.append(dict(dive=dive_id, model=m, b_cm=100 * HALF_THICKNESS_B_M.get(m, np.nan),
                              phi_sf_deg=np.degrees(phi_m), phi_anchor_deg=np.degrees(phi_anchor),
                              phi_sf_corrected_deg=np.degrees(phi_m_corr)))
pm = pd.DataFrame(per_model)
pm["sf_minus_anchor"] = pm.phi_sf_deg - pm.phi_anchor_deg
pm["corrected_minus_anchor"] = pm.phi_sf_corrected_deg - pm.phi_anchor_deg
print("per model, scale-free phi minus anchor phi (deg): median and spread over dives")
print(pm.groupby("model").agg(b_cm=("b_cm", "first"), cells=("dive", "size"),
                              raw_median=("sf_minus_anchor", "median"), raw_mad=("sf_minus_anchor", lambda s: (s - s.median()).abs().median()),
                              corrected_median=("corrected_minus_anchor", "median"), corrected_mad=("corrected_minus_anchor", lambda s: (s - s.median()).abs().median()),
                              ).sort_values("b_cm").round(3).to_string())
print()
print("all cells: |phi_sf - phi_anchor| median %.3f deg raw -> %.3f deg parallax-corrected;  within 0.05 deg: %d/%d raw -> %d/%d corrected" % (
    pm.sf_minus_anchor.abs().median(), pm.corrected_minus_anchor.abs().median(),
    (pm.sf_minus_anchor.abs() < 0.05).sum(), len(pm), (pm.corrected_minus_anchor.abs() < 0.05).sum(), pm.corrected_minus_anchor.notna().sum()))


per model, scale-free phi minus anchor phi (deg): median and spread over dives
              b_cm  cells  raw_median  raw_mad  corrected_median  corrected_mad
model                                                                          
Snook         -2.4      1       0.100    0.000            -0.055          0.000
Weasly Fish   -1.8     11       0.080    0.053            -0.005          0.032
Box           -0.4     10       0.003    0.016            -0.035          0.020
Purple Angel   NaN      4       0.028    0.014               NaN            NaN
Ruler          NaN      1      -0.043    0.000               NaN            NaN
Shark          NaN      2       0.268    0.178               NaN            NaN

all cells: |phi_sf - phi_anchor| median 0.060 deg raw -> 0.043 deg parallax-corrected;  within 0.05 deg: 13/29 raw -> 12/22 corrected


## §4.1 — does the checkerboard-vs-slate gap survive a target-free laser calibration?

P1 §6.3: inside the accuracy cohort the checkerboard-calibrated dives sit ~1.5 pp below
the slate-borrowed ones and the polish cannot separate calibration method from target
set. Recalibrating every dive from its own rigid objects would remove the laser target
from both groups.

**As run below, the test is not clean**, and the number should not be quoted: the
per-dive φ it uses is the joint fit over all models, which the previous section shows
is contaminated by thickness parallax on every solid model — and the two groups have
different model sets (the 2023 slate dives carry Grouper/Snook/Shark, the 2025
checkerboard dives Box/Weasly), so the contamination differs by group. A clean version
needs a parallax-free size on both sides: Box-only cells, or body height at the dot,
which this corpus does not record. Left in place so the next person does not re-run it
expecting an answer.


In [6]:
cohort = [d for d in cal.CORPUS_ACCURACY_DIVES if d in res.index]
c = res.loc[cohort]
for label, g in (("slate-borrowed", c[c.borrowed]), ("checkerboard self-calibrated", c[~c.borrowed])):
    print("%-30s n=%2d   recorded median %+.2f %%   scale-free median %+.2f %%" % (
        label, len(g), g.recorded_pct.median(), g.scalefree_pct.median()))
gap_rec = c[~c.borrowed].recorded_pct.median() - c[c.borrowed].recorded_pct.median()
gap_sf = c[~c.borrowed].scalefree_pct.median() - c[c.borrowed].scalefree_pct.median()
print("\ngap (checkerboard - slate): recorded %+.2f pp   after target-free laser recalibration %+.2f pp" % (gap_rec, gap_sf))


slate-borrowed                 n= 8   recorded median -0.15 %   scale-free median -1.04 %
checkerboard self-calibrated   n= 3   recorded median -3.17 %   scale-free median -3.53 %

gap (checkerboard - slate): recorded -3.02 pp   after target-free laser recalibration -2.49 pp


## What this does and does not show

**Established.** On the Box — the one object in the corpus whose dot and landmarks are
co-planar — the scale-free estimator reproduces P1's known-length calibration to
**0.003° median, 0.016° MAD, over ten dives**, spending no known length. That is the
P4 pool result (0.3 px on the board, 4 px on the SAM3-masked decoy by body height)
reproduced on production dives. The slate can leave the dive; so can the checkerboard,
for the laser.

**Established, the other way.** On solid fish models measured snout-to-fork the
estimator is biased by exactly the half-thickness parallax P1 §6.4 documents: the
per-model disagreement orders by `b` (Box 0.003°, trout 0.08°, Snook 0.10°, Shark
0.27°), and P1's parallax correction pulls the trout onto the anchor (0.08° → −0.005°).
A scale-free estimator on length cannot distinguish a thick object from a rotated
laser. This is the same lesson the P4 decoy taught with length vs height.

**Protocol consequence for P2.** The size fed to a target-free laser calibration must be
measured *at the dot's surface*: body height at the dot (SAM3 gives it), or a slab-like
object. A commodity brick target is a slab. It therefore needs **no known dimension**
for the laser — the laser half of §4.1's target comparison disappears rather than
needing to be graded.

**A note that belongs to P1.** The same parallax puts a genuine `+b/z²` range trend on
every thick model under a *correct* calibration (Grouper: ~+1.6 %/m over 1–3 m against
a 2 %/m flag). P1's range-trend audit is exposed to it on thick models; the Box is not.

**Limits.** The estimator inherits the range-trend gates, so a dive whose objects were
all shot at one range cannot be calibrated this way — the same dives P1 cannot audit;
P2's protocol should say "photograph something rigid at two ranges". Known lengths were
spent only in the validation columns and the parallax mechanism test; every φ in the
per-dive table is unchanged if they are deleted from the corpus.
